# Phase 3 — Modélisation baseline (XGBoost, FD001)

Objectif de cette phase : obtenir un premier modèle entraîné pour de vrai, avec un RMSE et un score NASA
mesurés sur le jeu de test officiel (`test_FD001.txt` + `RUL_FD001.txt`), reproductibles avec le code
de ce notebook — contrairement aux chiffres de l'ancien notebook exploratoire, supprimé
(voir [ADR-0006](decisions/0006-suppression-baseline-exploratoire.md) et la mise en garde dans le README).

## 1. Imports

On réutilise `src/preprocessing.py`, qui reprend exactement les fonctions déjà validées dans les
notebooks de Phase 1 (chargement, capteurs constants, normalisation, lissage) et de Phase 2
(calcul de la RUL, plafonnement) — pour ne pas recopier une 3e fois cette logique.

In [ ]:
import sys
sys.path.insert(0, "..")  # pour pouvoir importer src/ depuis notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul, DATA_DIR,
)

FENETRE = 5      # taille de la fenetre glissante, cf. Phase 1 (lissage) et README (benchmark)
PLAFOND_RUL = 125  # plafond de la RUL, cf. Phase 2


## 2. Pipeline Phases 1+2 sur le jeu d'entraînement

On enchaîne les étapes déjà validées : suppression des capteurs constants, normalisation min-max,
lissage par moyenne mobile (moteur par moteur), calcul de la RUL, puis plafonnement à 125 cycles.
Le `scaler` (min-max) est conservé : il faudra l'appliquer tel quel au jeu de test plus tard,
sans le ré-entraîner dessus (sinon fuite d'information entre train et test).

In [ ]:
train = load_fd001("train")  # chargement du fichier d'entrainement FD001

sensor_cols = get_sensor_columns(train.columns)  # les 21 capteurs bruts
constant_sensors = find_constant_sensors(train, sensor_cols)  # capteurs a variance nulle (Phase 1)
train = train.drop(columns=constant_sensors)  # on les retire
sensor_cols_restants = get_sensor_columns(train.columns)  # les 15 capteurs informatifs restants

train, scaler = normalize_minmax(train, sensor_cols_restants)  # normalisation min-max (0 a 1), scaler garde pour le test
train = lisser_capteurs(train, sensor_cols_restants, fenetre=FENETRE)  # moyenne mobile, moteur par moteur
train = calculer_rul(train)  # RUL brute par cycle
train = plafonner_rul(train, plafond=PLAFOND_RUL)  # RUL plafonnee a 125

train.shape  # verification rapide : 20631 lignes, colonnes = 26 - 6 constants + RUL + RUL_cappee = 22


## 3. Features par fenêtre glissante

Une valeur de capteur isolée ne dit pas si le moteur est stable ou en train de se dégrader. On ajoute donc,
pour chaque capteur et chaque cycle, deux nouvelles colonnes calculées sur les 5 derniers cycles
(fenêtre glissante, comme le lissage de la Phase 1, mais ici pour créer des features, pas pour nettoyer le signal) :

- **moyenne glissante** : la tendance récente du capteur.
- **écart-type glissant** : sa dispersion récente — en Phase 2, on a prouvé que cette dispersion double ou
  triple en fin de vie (`sensor_11`) : c'est exactement ce que cette feature permet au modèle de voir.

La fenêtre "glisse" d'un cycle à l'autre : au cycle *t*, elle regarde les cycles *t-4* à *t* (jamais le futur),
et jamais au-delà du début de vie d'un même moteur (calcul refait moteur par moteur, comme le lissage).

In [ ]:
def ajouter_features_glissantes(df, sensor_cols, fenetre):
    """
    Ajoute, pour chaque capteur, deux colonnes calculees sur une fenetre glissante de cycles :
    la moyenne et l'ecart-type des `fenetre` derniers cycles, moteur par moteur.

    Parameters
    ----------
    df : pandas.DataFrame
        Le tableau source (n'est pas modifie sur place, une copie est renvoyee).
    sensor_cols : list de str
        Les noms de colonnes capteurs pour lesquelles calculer les features.
    fenetre : int
        Nombre de cycles utilises pour la fenetre glissante.

    Returns
    -------
    pandas.DataFrame
        Le tableau avec 2 nouvelles colonnes par capteur : "<capteur>_glissante_moy" et "<capteur>_glissante_std".
    """
    df = df.copy()  # copie independante, pour ne pas modifier df sur place

    for sensor_name in sensor_cols:  # on traite chaque capteur, un par un
        colonne_moyenne = sensor_name + "_glissante_moy"
        colonne_ecart_type = sensor_name + "_glissante_std"
        df[colonne_moyenne] = 0.0  # valeurs provisoires, remplacees ligne par ligne juste apres
        df[colonne_ecart_type] = 0.0

        for moteur_id in df["unit_number"].unique():  # on traite chaque moteur separement
            masque = df["unit_number"] == moteur_id  # Vrai sur les lignes de ce moteur, Faux sinon
            # rolling() est causal par defaut (ne regarde que le passe) et ne porte que sur ce moteur
            df.loc[masque, colonne_moyenne] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).mean()
            df.loc[masque, colonne_ecart_type] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).std()

    return df


train = ajouter_features_glissantes(train, sensor_cols_restants, FENETRE)

# apercu sur le moteur 1 : la valeur brute lissee, sa moyenne glissante et son ecart-type glissant
colonnes_apercu = ["time_cycles", "sensor_11", "sensor_11_glissante_moy", "sensor_11_glissante_std"]
train.loc[train["unit_number"] == 1, colonnes_apercu].head(8)


## 4. Cas particulier : le tout premier cycle de chaque moteur

Au tout premier cycle d'un moteur, la fenêtre glissante ne contient qu'**une seule valeur** — l'écart-type
d'une seule valeur n'existe pas mathématiquement (il faut au moins 2 points pour mesurer une dispersion),
`pandas` renvoie donc `NaN` à cet endroit précis. Ça concerne exactement 1 ligne par moteur par capteur,
soit 100 moteurs × 15 capteurs = 1500 valeurs sur les 20 631 lignes du tableau.

On remplace ces `NaN` par 0 : au tout premier cycle, on n'a encore observé aucune variation, donc une
dispersion de 0 est la valeur la plus honnête qu'on puisse donner au modèle (pas une approximation
hasardeuse, une absence d'information réelle).

In [ ]:
# noms des colonnes "ecart-type glissant" : on parcourt toutes les colonnes du tableau et on garde
# celles dont le nom finit par "_glissante_std"
colonnes_ecart_type = []
for colonne in train.columns:
    if colonne.endswith("_glissante_std"):
        colonnes_ecart_type.append(colonne)

nb_nan_avant = train[colonnes_ecart_type].isna().sum().sum()  # total de NaN sur toutes les colonnes d'ecart-type
train[colonnes_ecart_type] = train[colonnes_ecart_type].fillna(0.0)  # remplace ces NaN par 0
nb_nan_apres = train[colonnes_ecart_type].isna().sum().sum()

print(f"NaN sur les ecarts-types glissants : {nb_nan_avant} avant remplissage, {nb_nan_apres} apres.")
assert nb_nan_avant == 100 * len(sensor_cols_restants), (
    f"attendu {100 * len(sensor_cols_restants)} NaN (1 par moteur et par capteur, au 1er cycle), trouve {nb_nan_avant}"
)
assert nb_nan_apres == 0, f"il reste {nb_nan_apres} NaN apres le fillna"


## 5. Construction des features et de la cible

Les features (`X`) sont les 15 capteurs lissés + leurs 15 moyennes glissantes + leurs 15 écarts-types
glissants + `time_cycles` (le numéro du cycle en cours), soit **46 colonnes**. La cible (`y`) est
`RUL_cappee` (Phase 2). `groups` retient le numéro de moteur de chaque ligne — indispensable pour la
validation croisée par moteur juste après.

**`time_cycles`** a été ajouté après un test d'ablation (section 11, en bas du notebook) : en validation
croisée, il améliore nettement le RMSE et le score NASA. Il est connu au moment de prédire (sur le test,
c'est le dernier cycle observé du moteur) et n'est pas normalisé (XGBoost est insensible à l'échelle).
**`unit_number`**, lui, reste exclu : c'est une étiquette de moteur, pas une mesure, et un nouveau moteur
n'a pas de numéro qui ait un sens pour le modèle.

In [ ]:
# noms des colonnes "moyenne glissante" : on parcourt toutes les colonnes du tableau et on garde
# celles dont le nom finit par "_glissante_moy"
colonnes_moyenne = []
for colonne in train.columns:
    if colonne.endswith("_glissante_moy"):
        colonnes_moyenne.append(colonne)

# 15 capteurs lisses + 15 moyennes + 15 ecarts-types + le numero de cycle en cours = 46 colonnes.
# time_cycles est ajoute apres le test d'ablation de la section 11. unit_number reste exclu : c'est
# une etiquette de moteur, pas une mesure (un nouveau moteur n'a pas de numero qui ait un sens).
feature_cols = sensor_cols_restants + colonnes_moyenne + colonnes_ecart_type + ["time_cycles"]

X = train[feature_cols]        # les 46 colonnes de features (entrees du modele)
y = train["RUL_cappee"]        # la cible a predire (Phase 2)

# groups sert UNIQUEMENT a GroupKFold plus bas, pour savoir quelles lignes appartiennent au meme moteur.
# scikit-learn ne connait pas le nom "unit_number" : on lui donnera juste cette suite de nombres
# (1,1,1,...,2,2,2,...,100,100), sans aucun contexte. C'est NOUS qui choisissons cette colonne
# precisement parce qu'elle represente "quel moteur" - si on se trompait de colonne ici (par exemple
# time_cycles), le code tournerait quand meme, sans erreur, mais grouperait n'importe quoi.
groups = train["unit_number"]

print(f"Nombre de features : {len(feature_cols)} ({len(sensor_cols_restants)} capteurs lisses + {len(colonnes_moyenne)} moyennes + {len(colonnes_ecart_type)} ecarts-types + time_cycles)")
print(f"X : {X.shape}, y : {y.shape}")

## 6. Le score NASA (Saxena et al., 2008)

Le RMSE traite une surestimation et une sous-estimation de la RUL de façon symétrique. Or le coût réel
ne l'est pas : sous-estimer la RUL entraîne une maintenance un peu prématurée (coût faible), tandis que
la surestimer risque une panne en vol (coût élevé). Le score NASA formalise cette asymétrie :

- si `d = RUL_prédite - RUL_vraie` est **négatif** (sous-estimation, prudent) : pénalité `exp(-d/13) - 1`
- si `d` est **positif** (surestimation, dangereux) : pénalité `exp(d/10) - 1` — croît beaucoup plus vite

Exemple : un écart de 20 cycles coûte `exp(20/13)-1 ≈ 3.66` en sous-estimation, mais `exp(20/10)-1 ≈ 6.39`
en surestimation — near le double, pour le même écart en valeur absolue.

In [ ]:
def score_nasa(y_true, y_pred):
    """
    Calcule le score NASA (Saxena et al., 2008), asymetrique : penalise plus fortement
    une surestimation de la RUL (dangereux) qu'une sous-estimation (prudent).

    Parameters
    ----------
    y_true : array-like
        Les vraies valeurs de RUL.
    y_pred : array-like
        Les valeurs predites par le modele.

    Returns
    -------
    float
        La somme des penalites individuelles sur tous les exemples.
    """
    total = 0.0  # on cumule la penalite de chaque exemple

    for vrai, predit in zip(y_true, y_pred):  # boucle explicite, un exemple a la fois
        d = predit - vrai  # positif si on surestime, negatif si on sous-estime

        if d < 0:  # sous-estimation : penalite plus douce
            total += np.exp(-d / 13) - 1
        else:  # surestimation : penalite plus severe
            total += np.exp(d / 10) - 1

    return total


# sanity check avant utilisation reelle : on verifie la fonction sur un exemple ou le calcul a la main est simple
test_surestimation = score_nasa([100], [120])  # d = +20
test_sousestimation = score_nasa([100], [80])  # d = -20
print(f"surestimation +20 : {test_surestimation:.3f} (attendu {np.exp(2) - 1:.3f})")
print(f"sous-estimation -20 : {test_sousestimation:.3f} (attendu {np.exp(20 / 13) - 1:.3f})")
assert abs(test_surestimation - (np.exp(2) - 1)) < 0.01
assert abs(test_sousestimation - (np.exp(20 / 13) - 1)) < 0.01


## 7. Validation croisée par moteur (GroupKFold)

Si on répartissait les lignes au hasard entre entraînement et validation, des cycles du **même moteur**
pourraient se retrouver des deux côtés. Le modèle apprendrait alors en partie à reconnaître ce moteur
précis plutôt qu'un patron général de dégradation — une fuite d'information qui rendrait le score de
validation artificiellement optimiste, sans rapport avec la performance réelle sur un moteur jamais vu.

`GroupKFold` avec `unit_number` comme groupe garantit qu'un moteur donné va **entièrement** d'un côté
(train) ou de l'autre (validation), jamais coupé en deux. On vérifie ça explicitement à chaque pli.

**`shuffle=True, random_state=42`** : par défaut, `GroupKFold` répartit les moteurs entre les plis selon
l'ordre où ils apparaissent dans le fichier, pas au hasard — si cet ordre correspondait à autre chose
qu'un simple ordre de simulation (par exemple des moteurs triés par robustesse), ça pourrait biaiser
chaque pli sans qu'on le sache. Vérifié : avec `shuffle=True`, le RMSE moyen ne change presque pas
(18.61 sans mélange vs 18.64 avec) — donc pas de biais cachés ici — mais on garde le mélange (avec une
graine fixe pour rester reproductible) pour ne plus jamais avoir à supposer que l'ordre du fichier n'a
pas d'importance.

In [ ]:
# gkf.split(...) est une fonction de la bibliotheque scikit-learn (pas notre code) : c'est A L'INTERIEUR
# d'elle, invisible ici, qu'elle regarde les valeurs uniques de "groups" (les 100 moteurs), decide
# comment les repartir en 5 paquets (melanges grace a shuffle=True, random_state=42 pour rester
# reproductible), puis traduit chaque paquet de moteurs en liste de numeros de ligne. On ne voit que
# le resultat (idx_train, idx_val), jamais le mecanisme interne - d'ou la verification juste en dessous.
gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
rmse_par_fold = []
rmse_train_par_fold = []  # erreur sur les moteurs APPRIS, pour le diagnostic biais/variance (section 12)
nasa_par_fold = []

for i, (idx_train, idx_val) in enumerate(gkf.split(X, y, groups=groups)):  # 5 plis
    # idx_train / idx_val = juste des listes de numeros de ligne pour ce tour precis,
    # differentes a chaque iteration de la boucle
    X_tr, X_val = X.iloc[idx_train], X.iloc[idx_val]
    y_tr, y_val = y.iloc[idx_train], y.iloc[idx_val]

    # verification explicite qu'aucun moteur n'apparait des deux cotes de ce pli : on ne fait pas
    # confiance aveuglement au mecanisme cache de GroupKFold, on verifie le resultat nous-memes
    moteurs_train = set(groups.iloc[idx_train])
    moteurs_val = set(groups.iloc[idx_val])
    assert len(moteurs_train & moteurs_val) == 0, "un moteur est present des deux cotes !"

    # un modele TOUT NEUF a chaque tour (recree dans la boucle) : aucune memoire du tour precedent
    modele = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
    modele.fit(X_tr, y_tr)           # apprend uniquement sur les moteurs de ce tour
    y_pred = modele.predict(X_val)   # predit sur des moteurs jamais vus par ce modele
    y_pred_train = modele.predict(X_tr)  # predit sur les moteurs qu'il vient d'apprendre (erreur train)

    # meme calcul de RMSE qu'a la main (racine de la moyenne des ecarts au carre)
    rmse = np.sqrt(np.mean((y_val.values - y_pred) ** 2))
    nasa = score_nasa(y_val.values, y_pred)
    rmse_train = np.sqrt(np.mean((y_tr.values - y_pred_train) ** 2))  # meme formule, cote train
    rmse_par_fold.append(rmse)  # on stocke pour faire la moyenne des 5 tours a la fin
    rmse_train_par_fold.append(rmse_train)
    nasa_par_fold.append(nasa)

    print(f"Pli {i + 1} : {len(moteurs_train)} moteurs train / {len(moteurs_val)} moteurs val -> RMSE train={rmse_train:.2f}, RMSE val={rmse:.2f}, score NASA val={nasa:.1f}")

print(f"\nMoyenne sur les 5 plis : RMSE val={np.mean(rmse_par_fold):.2f}, score NASA val={np.mean(nasa_par_fold):.1f}")
print(f"RMSE train moyen (moteurs appris) : {np.mean(rmse_train_par_fold):.2f} -> interpretation en section 12")

## 8. Modèle final, entraîné sur les 100 moteurs

La validation croisée sert à estimer la performance de façon fiable — une fois qu'on lui fait confiance,
on entraîne un dernier modèle sur **l'intégralité** des 100 moteurs d'entraînement (plus de données =
meilleur modèle), pour l'évaluer ensuite sur le vrai jeu de test officiel.

In [ ]:
modele_final = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
modele_final.fit(X, y)
print("Modele final entraine sur les 100 moteurs d'entrainement.")


## 9. Évaluation sur le jeu de test officiel

`test_FD001.txt` contient des trajectoires **tronquées avant la panne** : on ne connaît la RUL réelle
qu'au tout dernier cycle enregistré de chaque moteur test, donnée séparément dans `RUL_FD001.txt`
(100 valeurs, une par moteur). On applique donc le pipeline au jeu de test, mais avec deux règles
strictes pour éviter toute fuite d'information avec le train :

1. **Mêmes capteurs constants retirés** (ceux identifiés sur le train, pas recalculés sur le test).
2. **Même `scaler`, en `transform` seul** (jamais `fit` à nouveau) — sinon le test influencerait
   la normalisation, ce qui n'est pas possible en conditions réelles (on ne voit jamais le test à l'avance).

In [ ]:
test = load_fd001("test")  # chargement du fichier de test FD001
test = test.drop(columns=constant_sensors)  # memes capteurs retires que sur le train

# transform seul (pas fit_transform) : on applique le scaler deja appris sur le train, sans le reajuster
test[sensor_cols_restants] = scaler.transform(test[sensor_cols_restants])

test = lisser_capteurs(test, sensor_cols_restants, fenetre=FENETRE)  # meme lissage, moteur par moteur
test = ajouter_features_glissantes(test, sensor_cols_restants, FENETRE)  # memes features glissantes
test[colonnes_ecart_type] = test[colonnes_ecart_type].fillna(0.0)  # meme traitement du 1er cycle

test.shape  # verification rapide : 13096 lignes (moteurs test, trajectoires plus courtes)


### Dernier cycle enregistré de chaque moteur test

C'est le seul point de chaque moteur pour lequel on a une RUL vraie fournie (`RUL_FD001.txt`) —
on ne peut donc évaluer le modèle qu'à cet instant précis, pas sur toute la trajectoire test.

In [ ]:
# pour chaque moteur, on garde la ligne dont time_cycles est maximal (dernier cycle enregistre)
indices_derniers_cycles = test.groupby("unit_number")["time_cycles"].idxmax()
dernier_cycle_par_moteur = test.loc[indices_derniers_cycles].sort_values("unit_number").reset_index(drop=True)

rul_test_vraie = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL_vraie"])

print(f"Derniers cycles : {dernier_cycle_par_moteur.shape[0]} moteurs, RUL vraies fournies : {rul_test_vraie.shape[0]} valeurs")
assert dernier_cycle_par_moteur.shape[0] == 100
assert rul_test_vraie.shape[0] == 100


In [ ]:
X_test = dernier_cycle_par_moteur[feature_cols]
y_test_vraie = rul_test_vraie["RUL_vraie"].values

y_test_pred = modele_final.predict(X_test)
y_test_pred = np.clip(y_test_pred, a_min=0, a_max=None)  # une RUL negative n'a pas de sens physique

rmse_test = np.sqrt(np.mean((y_test_vraie - y_test_pred) ** 2))
nasa_test = score_nasa(y_test_vraie, y_test_pred)

ecarts = y_test_pred - y_test_vraie  # positif = surestimation (dangereux)
nb_surestimes_20 = int(np.sum(ecarts > 20))  # nombre de moteurs surestimes de plus de 20 cycles

print(f"RMSE test officiel        : {rmse_test:.2f}")
print(f"Score NASA test officiel  : {nasa_test:.1f}")
print(f"Moteurs surestimes de plus de 20 cycles : {nb_surestimes_20}/100")


## 10. Validation visuelle : prédictions vs vraies RUL

Un nuage de points prédiction vs vérité, avec la diagonale idéale (prédiction = vérité) : plus les points
sont proches de la diagonale, meilleur est le modèle. Les points au-dessus de la diagonale sont les
surestimations (le cas dangereux).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_test_vraie, y_test_pred, alpha=0.6, label="moteurs test")
limite = max(y_test_vraie.max(), y_test_pred.max()) + 5
ax.plot([0, limite], [0, limite], linestyle="--", color="black", label="prediction = verite")

ax.set_xlabel("RUL vraie (RUL_FD001.txt)")
ax.set_ylabel("RUL predite (modele final)")
ax.set_title(f"Test officiel FD001 - RMSE={rmse_test:.2f}, score NASA={nasa_test:.1f}, surestimes>20={nb_surestimes_20}/100")
ax.legend()

plt.tight_layout()
plt.show()


## 11. Pourquoi ces features ? Test d'ablation

On compare 7 jeux de features avec **exactement la même validation croisée** (mêmes plis, mêmes réglages
XGBoost). Le jeu de test officiel n'est **pas** utilisé ici : choisir des features en regardant le test
reviendrait à tricher.

Lecture de la sortie ci-dessous :
- Les colonnes glissantes (moyennes, écarts-types) n'améliorent pas le RMSE moyen de façon mesurable :
  les écarts entre les jeux de 15, 30 et 45 features sont bien plus petits que le bruit d'un pli à
  l'autre (écart-type entre plis d'environ 1,3, contre 0,2 d'écart au plus entre ces jeux).
- `time_cycles` améliore nettement le RMSE et le score NASA.
- « 15 capteurs + `time_cycles` » (16 features) fait aussi bien que les 46 : tout le gain vient de
  `time_cycles`. Les 46 sont conservés (déjà en place, et le sujet demande des statistiques glissantes
  pour la baseline classique) ; passer à 16 features est une simplification possible en Phase 4.

Limites : (1) la validation croisée évalue **tous** les cycles des moteurs mis de côté, alors que le test
officiel n'évalue que le dernier cycle de chaque moteur — le gain peut ne pas se transférer à l'identique ;
(2) un seul run par jeu de features, sans répétition.

In [ ]:
# On compare 7 jeux de features avec EXACTEMENT la meme validation croisee (memes plis pour tous)
# et les memes reglages XGBoost, pour justifier le choix des 46 features.
configs = {
    "15 : capteurs seuls": sensor_cols_restants,
    "30 : capteurs + moyennes": sensor_cols_restants + colonnes_moyenne,
    "30 : capteurs + ecarts-types": sensor_cols_restants + colonnes_ecart_type,
    "45 : capteurs + moyennes + ecarts-types": sensor_cols_restants + colonnes_moyenne + colonnes_ecart_type,
    "16 : capteurs + time_cycles": sensor_cols_restants + ["time_cycles"],
    "31 : capteurs + ecarts-types + time_cycles": sensor_cols_restants + colonnes_ecart_type + ["time_cycles"],
    "46 : les 45 + time_cycles (retenu)": feature_cols,
}

# les plis sont calcules UNE fois et reutilises pour toutes les configurations : la comparaison
# ne depend donc pas d'un decoupage different d'une configuration a l'autre
plis = list(GroupKFold(n_splits=5, shuffle=True, random_state=42).split(X, y, groups=groups))

for nom, colonnes in configs.items():  # une configuration a la fois
    X_config = train[colonnes]  # seulement les colonnes de cette configuration
    rmse_plis = []
    nasa_plis = []

    for idx_train, idx_val in plis:  # les 5 memes plis a chaque fois
        modele = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
        modele.fit(X_config.iloc[idx_train], y.iloc[idx_train])
        y_pred = modele.predict(X_config.iloc[idx_val])

        rmse_plis.append(np.sqrt(np.mean((y.iloc[idx_val].values - y_pred) ** 2)))
        nasa_plis.append(score_nasa(y.iloc[idx_val].values, y_pred))

    print(f"{nom:42s} RMSE moyen={np.mean(rmse_plis):6.2f} (ecart-type entre plis {np.std(rmse_plis):.2f})   score NASA moyen={np.mean(nasa_plis):9.1f}")

## 12. Diagnostic biais / variance : le modèle sur-apprend-il ou sous-apprend-il ?

Le RMSE de validation (16.86) dit **combien** le modèle se trompe sur des moteurs jamais vus, pas
**pourquoi**. Pour le savoir, on le compare à l'erreur sur les moteurs que le modèle a appris (RMSE train) :

| RMSE train | RMSE validation | Diagnostic | Remède habituel |
|---|---|---|---|
| bas | nettement plus haut | **variance** (sur-apprentissage : appris par cœur) | modèle plus simple, plus de données, régularisation |
| haut | proche du train | **biais** (sous-apprentissage : modèle trop simple) | modèle plus riche, meilleures features |
| bas | proche du train | équilibre | — |

Exemple concret : un élève qui a 19/20 aux exercices déjà corrigés en classe mais 10/20 à l'examen a
appris les corrigés par cœur (variance). Un élève qui a 10/20 aux deux n'a pas compris le cours (biais).

Trois mesures, toutes en validation croisée (le jeu de test officiel n'est **pas** utilisé, comme en
section 11) :
1. RMSE train vs validation pour chaque pli, aux réglages actuels (calculé en section 7) ;
2. une **courbe de validation** : RMSE train et validation quand on fait varier la profondeur des arbres
   (`max_depth`), le réglage qui contrôle le plus directement la complexité du modèle ;
3. la même chose en fonction du nombre d'arbres (`n_estimators`).

Ces mesures servent à **diagnostiquer**, pas à changer les réglages ici : l'optimisation des
hyperparamètres est prévue en Phase 4.

In [ ]:
# 1) Ecart train / validation pli par pli, avec les reglages actuels (valeurs calculees en section 7)
print("Pli | RMSE train | RMSE val | ecart (val - train)")
for i in range(len(rmse_par_fold)):  # un pli a la fois
    ecart = rmse_par_fold[i] - rmse_train_par_fold[i]  # positif = le modele fait moins bien sur l'inconnu
    print(f" {i + 1}  |   {rmse_train_par_fold[i]:6.2f}   |  {rmse_par_fold[i]:6.2f}  |  {ecart:+.2f}")

rmse_train_moyen = np.mean(rmse_train_par_fold)  # erreur moyenne sur les moteurs appris
rmse_val_moyen = np.mean(rmse_par_fold)          # erreur moyenne sur les moteurs jamais vus
print(f"\nMoyenne : train={rmse_train_moyen:.2f}, val={rmse_val_moyen:.2f}, ecart={rmse_val_moyen - rmse_train_moyen:+.2f}")

# sanity check : un modele evalue sur ses propres donnees d'apprentissage doit faire au moins aussi bien
# que sur des moteurs inconnus. Si ce n'etait pas le cas, il y aurait un bug (mauvais index, mauvaise colonne).
for i in range(len(rmse_par_fold)):
    assert rmse_train_par_fold[i] < rmse_par_fold[i], f"pli {i + 1} : RMSE train >= RMSE val, a verifier"

# visualisation : deux barres par pli (train / validation), pour voir l'ecart d'un coup d'oeil
fig, ax = plt.subplots(figsize=(8, 4.5))
positions = np.arange(len(rmse_par_fold))  # 0, 1, 2, 3, 4 : une position par pli
largeur = 0.38                              # largeur de chaque barre

ax.bar(positions - largeur / 2, rmse_train_par_fold, width=largeur, label="RMSE train (moteurs appris)")
ax.bar(positions + largeur / 2, rmse_par_fold, width=largeur, label="RMSE validation (moteurs jamais vus)")

etiquettes = []
for i in range(len(rmse_par_fold)):
    etiquettes.append(f"Pli {i + 1}")
ax.set_xticks(positions)
ax.set_xticklabels(etiquettes)
ax.set_ylabel("RMSE (cycles)")
ax.set_title(f"Train vs validation, reglages actuels - moyenne train={rmse_train_moyen:.2f}, val={rmse_val_moyen:.2f}")
ax.set_ylim(0, 23)  # marge en haut pour que la legende ne cache aucune barre
ax.legend(loc="upper left", ncol=2)

plt.tight_layout()
plt.savefig("../docs/figures/03_biais_variance_train_vs_val_par_pli.png", dpi=120)  # outputs effaces par nbstripout
plt.show()

### Courbe de validation : profondeur des arbres (`max_depth`)

On refait la même validation croisée (les **mêmes 5 plis** que la section 11, variable `plis`) en ne
changeant qu'un seul réglage : la profondeur maximale des arbres, de 1 (arbres très simples) à 10 (arbres
très complexes). Tout le reste est identique (`n_estimators=200`, `learning_rate=0.05`, 46 features).

Lecture attendue : quand la complexité augmente, le RMSE train baisse toujours. Le RMSE validation baisse
d'abord (on réduit le biais), puis stagne ou remonte (la variance prend le dessus). Le bon réglage est
là où la validation est la plus basse ; l'écart entre les deux courbes mesure la variance.

In [ ]:
# 2) Courbe de validation sur max_depth : un seul reglage change, tout le reste est fixe
profondeurs = [1, 2, 3, 4, 5, 6, 7, 8, 10]  # 5 = reglage actuel
train_moy_par_profondeur = []  # RMSE train moyen sur les 5 plis, pour chaque profondeur
val_moy_par_profondeur = []    # RMSE validation moyen
val_std_par_profondeur = []    # ecart-type de la validation entre plis (= le bruit de la mesure)

for profondeur in profondeurs:  # une profondeur a la fois
    rmse_train_plis = []
    rmse_val_plis = []

    for idx_train, idx_val in plis:  # les 5 memes plis qu'en section 11
        modele = XGBRegressor(n_estimators=200, max_depth=profondeur, learning_rate=0.05, random_state=42)
        modele.fit(X.iloc[idx_train], y.iloc[idx_train])

        pred_train = modele.predict(X.iloc[idx_train])  # erreur sur les moteurs appris
        pred_val = modele.predict(X.iloc[idx_val])      # erreur sur les moteurs jamais vus
        rmse_train_plis.append(np.sqrt(np.mean((y.iloc[idx_train].values - pred_train) ** 2)))
        rmse_val_plis.append(np.sqrt(np.mean((y.iloc[idx_val].values - pred_val) ** 2)))

    train_moy_par_profondeur.append(np.mean(rmse_train_plis))
    val_moy_par_profondeur.append(np.mean(rmse_val_plis))
    val_std_par_profondeur.append(np.std(rmse_val_plis))
    print(f"max_depth={profondeur:2d} : RMSE train={np.mean(rmse_train_plis):6.2f}   RMSE val={np.mean(rmse_val_plis):6.2f} "
          f"(ecart-type entre plis {np.std(rmse_val_plis):.2f})   ecart={np.mean(rmse_val_plis) - np.mean(rmse_train_plis):+.2f}")

# sanity check : le reglage actuel (max_depth=5) doit redonner les chiffres de la section 7 (memes plis, memes reglages)
indice_actuel = profondeurs.index(5)
assert abs(val_moy_par_profondeur[indice_actuel] - np.mean(rmse_par_fold)) < 0.05, "max_depth=5 ne redonne pas la section 7"

# meilleure profondeur en validation (information pour la Phase 4, pas un changement de reglage ici)
indice_meilleur = int(np.argmin(val_moy_par_profondeur))
print(f"\nPlus bas RMSE val : max_depth={profondeurs[indice_meilleur]} ({val_moy_par_profondeur[indice_meilleur]:.2f}), "
      f"contre {val_moy_par_profondeur[indice_actuel]:.2f} au reglage actuel (max_depth=5)")

# visualisation : deux courbes + une bande d'incertitude (+/- 1 ecart-type entre plis) autour de la validation
val_bas = np.array(val_moy_par_profondeur) - np.array(val_std_par_profondeur)
val_haut = np.array(val_moy_par_profondeur) + np.array(val_std_par_profondeur)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(profondeurs, train_moy_par_profondeur, marker="o", label="RMSE train (moteurs appris)")
ax.plot(profondeurs, val_moy_par_profondeur, marker="o", label="RMSE validation (moteurs jamais vus)")
ax.fill_between(profondeurs, val_bas, val_haut, alpha=0.2, color="tab:orange", label="validation +/- 1 ecart-type entre plis")
ax.axvline(5, linestyle="--", color="grey", label="reglage actuel (max_depth=5)")

ax.set_xlabel("max_depth (complexite de chaque arbre)")
ax.set_ylabel("RMSE moyen sur 5 plis (cycles)")
ax.set_title("Courbe de validation XGBoost - profondeur des arbres")
ax.legend()

plt.tight_layout()
plt.savefig("../docs/figures/03_biais_variance_courbe_max_depth.png", dpi=120)
plt.show()

### Courbe de validation : nombre d'arbres (`n_estimators`)

XGBoost construit ses arbres l'un après l'autre, chacun corrigeant les erreurs des précédents. Avec
`eval_set`, il mesure le RMSE train et validation **après chaque arbre ajouté** : un seul entraînement
de 1000 arbres par pli donne donc toute la courbe de 1 à 1000 arbres. Réglages fixes : `max_depth=5`,
`learning_rate=0.05`.

In [ ]:
# 3) Courbe de validation sur n_estimators, en un seul entrainement de 1000 arbres par pli
NB_ARBRES_MAX = 1000
courbes_train = []  # une courbe (1000 valeurs de RMSE) par pli, cote train
courbes_val = []    # idem, cote validation

for idx_train, idx_val in plis:  # les 5 memes plis
    X_tr_pli, y_tr_pli = X.iloc[idx_train], y.iloc[idx_train]
    X_val_pli, y_val_pli = X.iloc[idx_val], y.iloc[idx_val]

    # eval_metric="rmse" : XGBoost calcule le RMSE sur chaque jeu de eval_set apres chaque arbre ajoute
    modele = XGBRegressor(n_estimators=NB_ARBRES_MAX, max_depth=5, learning_rate=0.05,
                          random_state=42, eval_metric="rmse")
    modele.fit(X_tr_pli, y_tr_pli, eval_set=[(X_tr_pli, y_tr_pli), (X_val_pli, y_val_pli)], verbose=False)

    historique = modele.evals_result()  # validation_0 = 1er jeu de eval_set (train), validation_1 = 2e (val)
    courbes_train.append(historique["validation_0"]["rmse"])
    courbes_val.append(historique["validation_1"]["rmse"])

# moyenne des 5 plis, arbre par arbre (np.mean sur l'axe 0 = moyenne "verticale" des 5 courbes)
courbe_train_moy = np.mean(np.array(courbes_train), axis=0)
courbe_val_moy = np.mean(np.array(courbes_val), axis=0)

# sanity check : apres 200 arbres (reglage actuel), on doit retrouver le RMSE val de la section 7
print(f"Apres 200 arbres : RMSE val moyen={courbe_val_moy[199]:.2f} (section 7 : {np.mean(rmse_par_fold):.2f})")
assert abs(courbe_val_moy[199] - np.mean(rmse_par_fold)) < 0.05, "la courbe ne redonne pas la section 7"

for nb_arbres in [50, 100, 200, 400, 700, 1000]:  # quelques points representatifs de la courbe
    ecart = courbe_val_moy[nb_arbres - 1] - courbe_train_moy[nb_arbres - 1]
    print(f"{nb_arbres:4d} arbres : RMSE train={courbe_train_moy[nb_arbres - 1]:6.2f}   RMSE val={courbe_val_moy[nb_arbres - 1]:6.2f}   ecart={ecart:+.2f}")

nb_arbres_meilleur = int(np.argmin(courbe_val_moy)) + 1  # +1 car l'indice 0 correspond au 1er arbre
print(f"\nPlus bas RMSE val : {nb_arbres_meilleur} arbres ({courbe_val_moy[nb_arbres_meilleur - 1]:.2f})")

fig, ax = plt.subplots(figsize=(8, 4.5))
nombre_arbres = np.arange(1, NB_ARBRES_MAX + 1)  # 1, 2, ..., 1000 pour l'axe horizontal
ax.plot(nombre_arbres, courbe_train_moy, label="RMSE train (moteurs appris)")
ax.plot(nombre_arbres, courbe_val_moy, label="RMSE validation (moteurs jamais vus)")
ax.axvline(200, linestyle="--", color="grey", label="reglage actuel (200 arbres)")

ax.set_xlabel("nombre d'arbres (n_estimators)")
ax.set_ylabel("RMSE moyen sur 5 plis (cycles)")
ax.set_ylim(0, 40)  # on coupe le debut de courbe (RMSE tres eleve avec 1-2 arbres) pour lire la zone utile
ax.set_title("Courbe de validation XGBoost - nombre d'arbres (max_depth=5)")
ax.legend()

plt.tight_layout()
plt.savefig("../docs/figures/03_biais_variance_courbe_n_estimators.png", dpi=120)
plt.show()

### Lecture des résultats

**Aux réglages actuels, le modèle sur-apprend un peu** : RMSE train 10.94 contre 16.86 en validation,
soit un écart de 5.9 cycles, présent sur les 5 plis (de +4.0 à +8.2). Les deux courbes de validation le
confirment : quand on augmente la complexité (arbres plus profonds, plus d'arbres), l'erreur train continue
de baisser (jusqu'à 1.84 avec `max_depth=10`) alors que l'erreur de validation remonte légèrement. Le
modèle actuel est donc du côté « variance » du point optimal, pas du côté « biais ».

**Mais ce sur-apprentissage coûte peu en pratique** :
- le meilleur réglage trouvé (`max_depth=3` : 16.55 ; ou 90 arbres : 16.72) ne gagne que 0.1 à 0.3 cycle
  de RMSE de validation, bien moins que le bruit d'un pli à l'autre (écart-type ≈ 1.2) ;
- la courbe de validation est presque plate entre `max_depth=2` et `max_depth=10` (16.55 à 17.32).

**Ce qui limite vraiment le modèle, c'est un plancher autour de 16.5 en validation**, que la complexité
ne fait pas baisser : même le meilleur réglage reste là. Ce plancher vient de ce que les features ne
contiennent pas (biais lié aux features) ou du bruit propre aux données (deux moteurs aux capteurs
semblables peuvent avoir des RUL différentes) — pas d'un modèle trop complexe. Les pistes pour le
dépasser sont donc des features ou des modèles différents (séquences complètes pour un LSTM/CNN 1D), pas
seulement un réglage plus prudent.

**Décision** : les réglages ne sont **pas** modifiés dans ce notebook (gain non significatif, et les
chiffres de la Phase 3 sont déjà validés). Un modèle plus simple (`max_depth=3`, ou moins d'arbres) est
noté comme piste pour l'optimisation des hyperparamètres de la Phase 4, à départager avec un vrai test
statistique.

## 13. Une erreur de 16.86 cycles, c'est grand ou petit ? Quatre références

Une erreur n'est jamais « grande » toute seule : il lui faut une référence. On en utilise quatre :

1. **L'échelle de la cible** : la RUL d'entraînement va de 0 à 125 cycles (plafond de la Phase 2).
   On exprime le RMSE en pourcentage de cette plage.
2. **Un modèle « bête » (baseline)** : il prédit toujours la RUL moyenne des moteurs d'entraînement,
   sans regarder les capteurs. Si notre modèle ne fait pas nettement mieux, il n'apporte rien.
3. **Le R²** : il fait cette comparaison à la baseline en un seul chiffre. `R² = 1 - MSE_modèle / variance(y)`
   (la variance de `y` est exactement le MSE de la baseline qui prédit la moyenne). R² proche de 1 : bien
   meilleur que la baseline ; proche de 0 : pas mieux que prédire la moyenne.
4. **Le besoin du sujet** : l'erreur acceptable pour une décision de maintenance. Le cahier des charges
   ne fixe **aucun seuil chiffré** ; la seule référence chiffrée sourcée du projet est le benchmark
   exploratoire publié dans le README (Random Forest, RMSE test officiel 18.97), à battre.

Comme dans les sections 11 et 12, la validation croisée utilise les mêmes 5 plis par moteur. Rien n'est
réentraîné ici : on réutilise les RMSE déjà mesurés en section 7 et le modèle final de la section 9.

In [ ]:
# 1) Echelle de la cible (RUL plafonnee du train)
PLAGE_RUL = PLAFOND_RUL  # la RUL plafonnee va de 0 a 125 cycles : la plage fait donc 125
print(f"Cible train : min={y.min():.0f}, max={y.max():.0f}, moyenne={y.mean():.2f}, ecart-type={y.std(ddof=0):.2f}")

# 2) et 3) Baseline et R2, pli par pli, sur les memes plis que la section 7
rmse_baseline_par_fold = []  # RMSE du modele "bete" (predit toujours la moyenne du train du pli)
r2_train_par_fold = []       # R2 du modele XGBoost sur les moteurs appris
r2_val_par_fold = []         # R2 du modele XGBoost sur les moteurs jamais vus

for i, (idx_train, idx_val) in enumerate(plis):  # memes plis que les sections 7, 11 et 12
    y_tr_pli = y.iloc[idx_train].values
    y_val_pli = y.iloc[idx_val].values

    # la baseline ne regarde aucun capteur : elle predit la meme valeur pour tout le monde,
    # la moyenne de la RUL des moteurs d'entrainement de ce pli (jamais celle des moteurs de validation)
    prediction_baseline = y_tr_pli.mean()
    rmse_baseline = np.sqrt(np.mean((y_val_pli - prediction_baseline) ** 2))
    rmse_baseline_par_fold.append(rmse_baseline)

    # R2 = 1 - MSE du modele / variance de y. On reutilise les RMSE de la section 7 (MSE = RMSE au carre).
    r2_train = 1 - rmse_train_par_fold[i] ** 2 / np.var(y_tr_pli)
    r2_val = 1 - rmse_par_fold[i] ** 2 / np.var(y_val_pli)
    r2_train_par_fold.append(r2_train)
    r2_val_par_fold.append(r2_val)

    print(f"Pli {i + 1} : baseline val={rmse_baseline:6.2f}   modele val={rmse_par_fold[i]:6.2f}   "
          f"R2 train={r2_train:.3f}   R2 val={r2_val:.3f}")

# meme verification que pour sklearn.metrics.r2_score, sur un pli, pour ne pas faire confiance a notre formule les yeux fermes
from sklearn.metrics import r2_score
idx_train_0, idx_val_0 = plis[0]
modele_verif = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
modele_verif.fit(X.iloc[idx_train_0], y.iloc[idx_train_0])
r2_sklearn = r2_score(y.iloc[idx_val_0], modele_verif.predict(X.iloc[idx_val_0]))
assert abs(r2_sklearn - r2_val_par_fold[0]) < 0.01, "notre formule du R2 ne colle pas avec scikit-learn"

# meme chose sur le test officiel : la baseline predit la RUL moyenne de TOUT le train pour chacun des 100 moteurs
rmse_baseline_test = np.sqrt(np.mean((y_test_vraie - y.mean()) ** 2))
r2_test = r2_score(y_test_vraie, y_test_pred)
BENCHMARK_EXPLORATOIRE_TEST = 18.97  # RMSE test officiel du Random Forest exploratoire (README, chiffre publie tel quel)

# tableau recapitulatif : chaque erreur avec ses references
lignes_tableau = [
    ("Validation croisee : baseline (moyenne)", np.mean(rmse_baseline_par_fold), None),
    ("Validation croisee : XGBoost, train", np.mean(rmse_train_par_fold), np.mean(r2_train_par_fold)),
    ("Validation croisee : XGBoost, validation", np.mean(rmse_par_fold), np.mean(r2_val_par_fold)),
    ("Test officiel : baseline (moyenne)", rmse_baseline_test, None),
    ("Test officiel : XGBoost", rmse_test, r2_test),
    ("Test officiel : benchmark exploratoire RF (README)", BENCHMARK_EXPLORATOIRE_TEST, None),
]
print(f"\n{'':52s} {'RMSE':>6s}  {'% de 0-125':>10s}  {'R2':>6s}")
for nom, rmse_ligne, r2_ligne in lignes_tableau:
    texte_r2 = "  -   "
    if r2_ligne is not None:
        texte_r2 = f"{r2_ligne:6.3f}"
    print(f"{nom:52s} {rmse_ligne:6.2f}  {100 * rmse_ligne / PLAGE_RUL:9.1f} %  {texte_r2}")

gain_val = np.mean(rmse_baseline_par_fold) / np.mean(rmse_par_fold)  # combien de fois mieux que la baseline
gain_test = rmse_baseline_test / rmse_test
print(f"\nXGBoost fait {gain_val:.1f} fois mieux que la baseline en validation croisee, {gain_test:.1f} fois mieux sur le test officiel.")

# visualisation : toutes les erreurs sur la meme echelle (0 a 125 cycles = la plage de la cible)
noms = []
valeurs = []
couleurs = []
for nom, rmse_ligne, r2_ligne in lignes_tableau:
    noms.append(nom)
    valeurs.append(rmse_ligne)
    if "baseline" in nom:
        couleurs.append("tab:grey")    # modele bete
    elif "benchmark" in nom:
        couleurs.append("tab:purple")  # reference a battre
    else:
        couleurs.append("tab:blue")    # notre modele

fig, ax = plt.subplots(figsize=(9, 4.5))
positions = np.arange(len(noms))
ax.barh(positions, valeurs, color=couleurs)
for position, valeur in zip(positions, valeurs):  # la valeur ecrite au bout de chaque barre
    ax.text(valeur + 0.5, position, f"{valeur:.2f} ({100 * valeur / PLAGE_RUL:.0f} %)", va="center")

ax.set_yticks(positions)
ax.set_yticklabels(noms)
ax.invert_yaxis()  # premiere ligne du tableau en haut
ax.set_xlim(0, 60)
ax.set_xlabel("RMSE (cycles) - entre parentheses : % de la plage de la cible (0 a 125)")
ax.set_title("Notre erreur comparee a la baseline et au benchmark")

plt.tight_layout()
plt.savefig("../docs/figures/03_echelle_erreur_baseline.png", dpi=120)  # outputs effaces par nbstripout
plt.show()

### Lecture des résultats

| Erreur | RMSE | % de la plage 0-125 | R² |
|---|---|---|---|
| Validation croisée : baseline (moyenne) | 41.69 | 33.4 % | — |
| Validation croisée : XGBoost, train | 10.94 | 8.8 % | 0.931 |
| Validation croisée : XGBoost, validation | 16.86 | 13.5 % | 0.835 |
| Test officiel : baseline (moyenne) | 43.07 | 34.5 % | — |
| Test officiel : XGBoost | 19.70 | 15.8 % | 0.775 |
| Test officiel : benchmark exploratoire RF (README) | 18.97 | 15.2 % | — |

1. **Échelle** : l'erreur de validation représente 13.5 % de la plage de la cible, 15.8 % sur le test.
2. **Baseline** : XGBoost fait 2.5 fois mieux que « prédire la moyenne » en validation croisée, 2.2 fois
   mieux sur le test. Le modèle apprend donc bien quelque chose des capteurs.
3. **R²** : 0.835 en validation (le modèle explique environ 84 % de la variance de la RUL), 0.775 sur le
   test. L'écart entre R² train (0.931) et validation (0.835) est la même variance que celle mesurée en
   section 12, vue sous un autre angle.
4. **Besoin du sujet** : sur le test officiel, notre XGBoost (19.70) **ne bat pas** le benchmark
   exploratoire (18.97), la seule référence chiffrée sourcée du projet. L'écart (0.73) n'est pas testé
   statistiquement et le test ne compte que 100 moteurs, mais l'objectif « battre la référence » n'est
   pas atteint à ce stade.

**Conclusion** : l'erreur est petite par rapport à la baseline (références 2 et 3), mais pas encore
suffisante par rapport au benchmark à battre (référence 4). Combiné à la section 12, on sait pourquoi :
le plancher d'environ 16.5 en validation ne vient pas des réglages.

Limites : le cahier des charges ne fixe aucune erreur acceptable chiffrée (à définir en Phase 4, à partir
d'une source vérifiée) ; sur le test, la vraie RUL n'est pas plafonnée (elle peut dépasser 125), ce qui
pénalise un peu la baseline comme le modèle ; un RMSE moyen ne dit rien de l'erreur près de la panne, là où
une erreur de 17 cycles pèse le plus (le score NASA en tient compte en partie).